1. Load master_dataset.parquet

2. Create outcome variables

3. Create DWEI pillars

4. Standardize indicators

5. Apply PCA

6. Build DWEI score

7. Rank districts

8. Save final modelling dataset

In [56]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

from scipy.stats import zscore

import joblib

In [57]:
master = pd.read_parquet(
    "../data/master/master_dataset.parquet"
)

print(master.shape)

(632, 46)


In [58]:
master["delta_STUNTING"] = (
    master["STUNTING_NFHS5"]
    - master["STUNTING_NFHS4"]
)

master["delta_WASTING"] = (
    master["WASTING_NFHS5"]
    - master["WASTING_NFHS4"]
)

master["delta_UNDERWEIGHT"] = (
    master["UNDERWEIGHT_NFHS5"]
    - master["UNDERWEIGHT_NFHS4"]
)

master["delta_ANEMIA"] = (
    master["ANEMIA_NFHS5"]
    - master["ANEMIA_NFHS4"]
)

master["delta_IMMUNIZATION"] = (
    master["IMMUNIZATION_NFHS5"]
    - master["IMMUNIZATION_NFHS4"]
)

master["delta_INST_DEL"] = (
    master["INST_DEL_NFHS5"]
    - master["INST_DEL_NFHS4"]
)

master["delta_SANITATION"] = (
    master["SANITATION_NFHS5"]
    - master["SANITATION_NFHS4"]
)

master["delta_CLEAN_FUEL"] = (
    master["CLEAN_FUEL_NFHS5"]
    - master["CLEAN_FUEL_NFHS4"]
)

In [59]:
delta_cols = [
    "delta_STUNTING",
    "delta_WASTING",
    "delta_UNDERWEIGHT",
    "delta_ANEMIA",
    "delta_IMMUNIZATION",
    "delta_INST_DEL",
    "delta_SANITATION",
    "delta_CLEAN_FUEL"
]

master[delta_cols].isna().sum()

delta_STUNTING         0
delta_WASTING          0
delta_UNDERWEIGHT      0
delta_ANEMIA           0
delta_IMMUNIZATION    17
delta_INST_DEL         0
delta_SANITATION       0
delta_CLEAN_FUEL       0
dtype: int64

In [60]:
need_features = [
    "female_literacy_pct",
    "scst_pct",
    "agri_worker_pct",
    "poverty_log",
    "night_lights_log"
]

In [62]:
master["poverty_log"] = (
    master.groupby("State")["poverty_log"]
    .transform(
        lambda x: x.fillna(x.median())
    )
)

In [63]:
master["poverty_log"] = (
    master["poverty_log"]
    .fillna(
        master["poverty_log"].median()
    )
)

In [64]:
master["poverty_log"].isna().sum()

np.int64(0)

In [65]:
scaler = StandardScaler()

master_scaled = master.copy()

master_scaled[need_features] = scaler.fit_transform(
    master[need_features]
)

In [66]:
master_scaled[
    need_features
].describe().T[
    ["mean", "std"]
]

,mean,std
female_literacy_pct,-2.304767e-16,1.000792
scst_pct,2.248553e-17,1.000792
agri_worker_pct,1.883163e-16,1.000792
poverty_log,-2.642050e-16,1.000792
night_lights_log,1.208597e-16,1.000792


In [68]:
joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

['../models/scaler.pkl']

In [69]:
ridge_results = {}

In [70]:
for outcome in delta_cols:

    temp = master_scaled[
        need_features + [outcome]
    ].dropna()

    X = temp[need_features]

    y = temp[outcome]

    model = Ridge(alpha=1.0)

    model.fit(X, y)

    y_pred = model.predict(X)

    residuals = y - y_pred

    master_scaled.loc[
        temp.index,
        f"residual_{outcome}"
    ] = residuals

    ridge_results[outcome] = {
        "model": model,
        "r2": r2_score(y, y_pred)
    }

In [71]:
r2_df = pd.DataFrame(
    {
        k: v["r2"]
        for k, v in ridge_results.items()
    },
    index=["R2"]
).T

r2_df

,R2
delta_STUNTING,0.102242
delta_WASTING,0.047136
delta_UNDERWEIGHT,0.106190
delta_ANEMIA,0.060427
delta_IMMUNIZATION,0.056714
delta_INST_DEL,0.268713
delta_SANITATION,0.336211
delta_CLEAN_FUEL,0.137723


In [72]:
for outcome in delta_cols:

    print(
        outcome,
        round(
            master_scaled[
                f"residual_{outcome}"
            ].mean(),
            10
        )
    )

delta_STUNTING -0.0
delta_WASTING 0.0
delta_UNDERWEIGHT -0.0
delta_ANEMIA -0.0
delta_IMMUNIZATION -0.0
delta_INST_DEL 0.0
delta_SANITATION 0.0
delta_CLEAN_FUEL -0.0


In [73]:
[c for c in master_scaled.columns if "residual" in c]

['residual_delta_STUNTING',
 'residual_delta_WASTING',
 'residual_delta_UNDERWEIGHT',
 'residual_delta_ANEMIA',
 'residual_delta_IMMUNIZATION',
 'residual_delta_INST_DEL',
 'residual_delta_SANITATION',
 'residual_delta_CLEAN_FUEL']

47. Flip sign
48. Z-score standardization
49. Create DWEI_score
50. Sanity checks
51. Save output

In [75]:
negative_residuals = [
    "residual_delta_STUNTING",
    "residual_delta_WASTING",
    "residual_delta_UNDERWEIGHT",
    "residual_delta_ANEMIA"
]

for col in negative_residuals:

    master_scaled[col] = (
        -1
        * master_scaled[col]
    )

In [76]:
residual_cols = [
    "residual_delta_STUNTING",
    "residual_delta_WASTING",
    "residual_delta_UNDERWEIGHT",
    "residual_delta_ANEMIA",
    "residual_delta_IMMUNIZATION",
    "residual_delta_INST_DEL",
    "residual_delta_SANITATION",
    "residual_delta_CLEAN_FUEL"
]

In [77]:
for col in residual_cols:

    master_scaled[
        f"z_{col}"
    ] = zscore(
        master_scaled[col],
        nan_policy="omit"
    )

In [78]:
z_cols = [
    col
    for col in master_scaled.columns
    if col.startswith("z_residual")
]

master_scaled[
    z_cols
].describe().T[
    ["mean", "std"]
]

,mean,std
z_residual_delta_STUNTING,2.248553e-17,1.000792
z_residual_delta_WASTING,-1.405346e-17,1.000792
z_residual_delta_UNDERWEIGHT,-2.248553e-17,1.000792
z_residual_delta_ANEMIA,1.124276e-17,1.000792
z_residual_delta_IMMUNIZATION,-2.310708e-17,1.000814
z_residual_delta_INST_DEL,3.372829e-17,1.000792
z_residual_delta_SANITATION,-2.248553e-17,1.000792
z_residual_delta_CLEAN_FUEL,-2.529622e-17,1.000792


In [79]:
master_scaled["DWEI_score"] = (
    master_scaled[
        z_cols
    ].mean(
        axis=1,
        skipna=True
    )
)

In [80]:
master_scaled["DWEI_score"].isna().sum()

np.int64(0)

In [81]:
master_scaled["DWEI_score"].describe()

count    632.000000
mean       0.000446
std        0.454707
min       -1.685924
25%       -0.275396
50%       -0.009838
75%        0.309347
max        1.230911
Name: DWEI_score, dtype: float64

In [82]:
master_scaled[
    [
        "State",
        "District",
        "DWEI_score"
    ]
].sort_values(
    "DWEI_score",
    ascending=False
).head(10)

,State,District,DWEI_score
274,Madhya Pradesh,Alirajpur,1.230911
22,Arunachal Pradesh,Lohit,1.156015
476,Rajasthan,Udaipur,1.099794
578,Uttar Pradesh,Jaunpur,1.074459
280,Madhya Pradesh,Bhind,1.055766
546,Uttar Pradesh,Auraiya,1.049080
370,Meghalaya,West Garo Hills,1.046632
530,Uttarakhand,Bageshwar,0.989172
595,Uttar Pradesh,Mirzapur,0.950429
547,Uttar Pradesh,Ayodhya,0.914991


In [84]:
master_scaled.to_parquet(
    "../data/master/master_dwei.parquet",
    index=False
)

master_scaled.to_csv(
    "../data/master/master_dwei.csv",
    index=False
)

✓ Delta creation
✓ Poverty handling
✓ StandardScaler
✓ Ridge models
✓ Residual generation
✓ Sign correction
✓ Z-score standardization
✓ DWEI score construction
✓ Validation
✓ Save outputs